# Interactive three-body problem: Namouni map and N-body (separate window, exhibition version)

The figure opens in **its own window** (Matplotlib's Tk backend), and the animation **keeps going until you pause it**. Text is large so it can be read on a screen from a distance; set `font_size` and `fig_size` in the parameters cell to suit the screen (the window can also be maximised).

Run all cells; the window appears (it may open behind the browser). Re-running the last cell replaces the window.

Panels:
1. **Left: surface of section from the Namouni map.** Click a point to start an orbit there; the **Namouni map** (`NamouniMap` from `testmap.ipynb`) adds that orbit's points in a new colour. The horizontal axis is the **conjunction angle** $-\theta$ (so the plot is the mirror image of the earlier $\theta$ versions), and the vertical axis is the **period ratio** $P_{\rm test}/P_{\rm planet} = (1 + \epsilon a_r)^{3/2}$, about 1.111 at the 10:9 resonance. The eccentricity for a clicked point comes from the conserved quantity $C = K - I$.
2. **Middle: N-body, in the frame rotating with the test body's pericentre.** The same click starts a **REBOUND N-body integration** (star, planet, massless test body) from the same initial condition. The star stays at the centre and the test body's pericentre always points straight up; the planet (white) and the test body are drawn with a one-orbit trail. A line from the star marks the direction of each conjunction: the latest is bright, the previous `Nlines - 1` are dim. The integration is extended as the animation needs it, so it runs until you press **Pause**, click a new orbit, press **Clear** or close the window. The dashed box (and the **+** at its centre) marks the region shown in the zoom panel: click to centre the box on a point, or press inside it and drag.
3. **Right: zoom.** The dashed box, magnified (same frame), with faint traces of the test body's earlier passes (from the last `Ntrace` conjunctions), so the change of its orbit at each conjunction shows up as a shift between groups of passes. Drag inside it to pan and scroll to zoom in or out about the cursor (the zoom slider follows).

The right two panels have a space background and are in astronomical units, with the planet's orbit at $a_p = 1$ AU.

Controls along the bottom: **Speed** (planet orbits per second) and **Zoom half-width** sliders, **Pause/Resume**, **Clear**, and a status line that shows the animation time and any error.

Since the right panels are now N-body, the test body's orbit changes smoothly through each encounter instead of jumping at a kick, and the rotating frame turns smoothly with its osculating pericentre.

Requires `tkinter` and `rebound` in the kernel environment (both are in `threebody`).

In [1]:
import time
import traceback
from functools import wraps
import numpy as np
import rebound
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.widgets import Button, Slider

import warnings #to catch warning in implicit update
warnings.simplefilter("error", RuntimeWarning)

## Model (copied from `testmap.ipynb`)
`NamouniMap` is unchanged, except that `runmap` passes `self.ind` to each step instead of the notebook-global `ind`.

In [2]:
class NamouniMap():
    r"""
    A class representing the Namouni map. The map depends on parameters,
    :math:`K`, :math:`\phi`, :math:`\lambda`, and :math:`I`. The map is defined by the equations

    .. math::
        \begin{align}
        I_{n+1} &= I_n + W_\phi(I_{n+1},K_{n+1},\lambda_n,\phi_n) \\
        \phi_{n+1} &= \phi_n + W_\I(I_{n+1},K_{n+1},\lambda_n,\phi_n) \\
        K_{n+1} = I_{n+1}-I_n+K_n \\
        \lambda_{n+1} = \lambda_n + \frac{4\pi}{3\epsilon\sqrt{8K_{n+1}/3}}
        \end{align}


    See also Namouni et al. (1996)

    Parameters
    ----------
    m : float
        Planet-star mass ratio.
    e0 : float
        Initial eccentricity of the test particle
    a_r0 : float
        Initial relative semimajor axis in Hill's unit
    lambda0 : float
        Initial conjunction longitude
    phi0 : float
        Initial pericenter longitude
    ind : int
        An integer that indicates the configuration of the planetary system. 
        If ind == 1, the massive planet comes before the test particle.
        Otherwise, the test particle comes first.
    exact_period : bool, optional
        If True, the update for lambda_{n+1} will be calculated as in Duncan et al. (1989), 
        which is the exact calculation.
        Otherwise, the update will follow the original Namouni mapping which is the approximation in Hill's limit.
        Default is True.
    """

    
    def __init__(self,m,ind,order,exact_period=True):
        self._m = m
        self._ind = ind
        # self._Nmax = Nmax
        # self._y0 = y0
        # self._update_amplitudes()
        # self.mod = mod
        self._order = order
        self.exact_period = exact_period

    @property
    def m(self):
        return self._m

    @property
    def ind(self):
        return self._ind

    @property
    def order(self):
        return self._order
    
    @property
    def epsilon(self):
        m = self._m
        return (m/3)**(1/3) 

    @property
    def exact_period(self):
        return self._exact_period
        
    @exact_period.setter
    def exact_period(self,val):
        self._exact_period = val
        if val:
            self._periodfn = lambda Knp1: 2*np.pi*np.abs(1/((1+np.sqrt(8*Knp1/3)*self.epsilon)**(-3/2)-1))
        else:
            self._periodfn = lambda Knp1: 4*np.pi/(3*self.epsilon*np.sqrt(8*Knp1/3))

    def namouni_fo_step(self, In, Kn, phin, lambdan, arn, ind):
        """First-order Namouni mapping step."""
        
        W101 = 3.359350011
        hn = np.sqrt(2*In)*np.cos(phin)
        kn = np.sqrt(2*In)*np.sin(phin)
        if ind == 1:
            hprime = hn + 2*W101*np.sin(lambdan)/arn**2 # + for Namouni, - for pomega
            kprime = kn - 2*W101*np.cos(lambdan)/arn**2 # - for Namouni, + for pomega
            # hprime = hn + 2*W101*np.sin(lambdan)/arn
            # kprime = kn - 2*W101*np.cos(lambdan)/arn
        else: #from DQT, the sign switches for the outer pair
            hprime = hn - 2*W101*np.sin(lambdan)/arn**2
            kprime = kn + 2*W101*np.cos(lambdan)/arn**2 
            # hprime = hn - 2*W101*np.sin(lambdan)/arn
            # kprime = kn + 2*W101*np.cos(lambdan)/arn
        phiprime = np.arctan2(kprime, hprime) # 4 quadrant arctan
        #print(hn, kn)
        # both options should give the same result:
        Iprime = 1/2*(hprime**2 + kprime**2)
        #Inprime = In + 2*W1/arn**2*np.sqrt(2*In)*np.sin(thetan) + 2*W1**2/arn**4
        #Namouni update
        hnp1 = hprime
        knp1 = kprime
        Inp1 = Iprime
        Knp1 = Kn + (Inp1 - In)
        # Knp1 = Kn + (Inp1 - In)*arn**2
        # print(Knp1)
        phinp1 = phiprime

        lambdanp1 = lambdan + self._periodfn(Knp1)
        # if exact_period==True:
        #     lambdanp1 = lambdan + 2*np.pi*np.abs(1/((1+np.sqrt(8*Knp1/3)*epsilon)**(-3/2)-1)) #same lambda
        # else:
        #     lambdanp1 = lambdan + 4*np.pi/(3*epsilon*np.sqrt(8*Knp1/3))
        return Inp1, Knp1, phinp1, lambdanp1, hnp1, knp1


    def namouni_so_step(self, In, Kn, phin, lambdan, arn, ind):
        """Second-order Namouni mapping step."""
        """The second order mapping involves an intermediate step and implicit scheme for eccentricity update."""
        """This calculates the implicit update iteratively instead of explicitly solving for the next value."""
        W200= -1
        W101= 3.359350011
        W202= -2.678665722        
        hn = np.sqrt(2*In)*np.cos(phin)
        kn = np.sqrt(2*In)*np.sin(phin)
        if ind == 1:
            hprime = hn + 2*W101*np.sin(lambdan)/arn**2 # + for Namouni, - for pomega
            kprime = kn - 2*W101*np.cos(lambdan)/arn**2 # - for Namouni, + for pomega
            # hprime = hn + 2*W101*np.sin(lambdan)/arn
            # kprime = kn - 2*W101*np.cos(lambdan)/arn
        else: #from DQT, the sign switches for the outer pair
            hprime = hn - 2*W101*np.sin(lambdan)/arn**2
            kprime = kn + 2*W101*np.cos(lambdan)/arn**2 
            # hprime = hn - 2*W101*np.sin(lambdan)/arn
            # kprime = kn + 2*W101*np.cos(lambdan)/arn
        phiprime = np.arctan2(kprime, hprime) # 4 quadrant arctan
        #thetaprime = (pomegaprime - lambdan)
        thetaprime = (phiprime - lambdan)
    
        # A = calc_coeff(2, W2, arn, thetaprime)
        # E1 = 1/(1+A)
        # Inp1 = E1*Inprime
        #implicit I update
        Iprime = 0.5*(hprime**2 + kprime**2)
        
        Inp1_prev = Iprime #first guess = Iprime
        while True: #iteratively update using implicit scheme
            Inp1_next = Iprime - 4*W202*np.sin(2*thetaprime)*2*Inp1_prev/arn**3
            # Inp1_next = Iprime - 4*W202*np.sin(2*thetaprime)*2*Inp1_prev/arn #with Istar scale
            if np.abs((Inp1_next-Inp1_prev)/Inp1_next) < 1e-12: #break when satisfied
                break
            Inp1_prev = Inp1_next
        # Inp1 = (0.5*(hprime**2 + kprime**2)) / (1 + 4*W2*np.sin(2*thetaprime)*2/arn**3)
        Inp1 = Inp1_next
    
        #pomeganp1 = pomegaprime - (W200 + 2*W2*np.cos(2*thetaprime))*2/arn**3
        phinp1 = phiprime - (W200 + 2*W202*np.cos(2*thetaprime))*2/arn**3
        Knp1 = Kn + Inp1 - In
        # Knp1 = Kn + (Inp1 - In)*arn**2 #Istar scale
        hnp1 = np.sqrt(2*Inp1)*np.cos(phinp1)
        knp1 = np.sqrt(2*Inp1)*np.sin(phinp1)
        lambdanp1 = lambdan + self._periodfn(Knp1)
        # if exact_period==True:
        #     lambdanp1 = lambdan + 2*np.pi*np.abs(1/((1+np.sqrt(8*Knp1/3)*epsilon)**(-3/2)-1)) #same lambda
        # else:
        #     lambdanp1 = lambdan + 4*np.pi/(3*epsilon*np.sqrt(8*Knp1/3))
        return Inp1, Knp1, phinp1, lambdanp1, hnp1, knp1



    def namouni_4o_step(self, In, Kn, phin, lambdan, arn, ind):
        """Fourth-order Namouni mapping step."""
        """The fourth order mapping involves an intermediate step and implicit scheme for eccentricity update."""
        """This calculates the implicit update iteratively instead of explicitly solving for the next value."""
        W200, W400= -1,-3/8
        W101, W301 = 3.359350011, 0.1430675142
        W202, W402 = -2.678665722, 0.5232506456
        W303 = 2.639498803
        W404 = -2.855951527
        hn = np.sqrt(2*In)*np.cos(phin)
        kn = np.sqrt(2*In)*np.sin(phin)
   
        if ind == 1:
            hprime = hn + 2*W101*np.sin(lambdan)/arn**2 # + for Namouni, - for pomega
            kprime = kn - 2*W101*np.cos(lambdan)/arn**2 # - for Namouni, + for pomega
            # hprime = hn + 2*W101*np.sin(lambdan)/arn
            # kprime = kn - 2*W101*np.cos(lambdan)/arn
        else: #from DQT, the sign switches for the outer pair
            hprime = hn - 2*W101*np.sin(lambdan)/arn**2
            kprime = kn + 2*W101*np.cos(lambdan)/arn**2 
            # hprime = hn - 2*W101*np.sin(lambdan)/arn
            # kprime = kn + 2*W101*np.cos(lambdan)/arn
        phiprime = np.arctan2(kprime, hprime) # 4 quadrant arctan
        #thetaprime = (pomegaprime - lambdan)
        thetaprime = (phiprime - lambdan)
    
        # A = calc_coeff(2, W2, arn, thetaprime)
        # E1 = 1/(1+A)
        # Inp1 = E1*Inprime
        #implicit I update
        Iprime = 0.5*(hprime**2 + kprime**2)
        
        Inp1_prev = Iprime #first guess = Iprime
        phinp1_prev = phiprime
        try:
            while True: #iteratively update using implicit scheme
                Inp1o1 = 2*(W301/arn**4*(2*Inp1_prev)**1.5)*np.sin(thetaprime)
                Inp1o2 = 4*W202*np.sin(2*thetaprime)*2*Inp1_prev/arn**3
                Inp1o3 = (2*W301*np.sin(thetaprime)+6*W303*np.sin(3*thetaprime))*(2*Inp1_prev)**(3/2)/arn**4
                Inp1o4 = (4*W402*np.sin(2*thetaprime)+8*W404*np.sin(4*thetaprime))*(2*Inp1_prev)**(2)/arn**5

                # #Istar scale
                # Inp1o1 = 2*(W301/arn*(2*Inp1_prev)**1.5)*np.sin(thetaprime)
                # Inp1o2 = 4*W202*np.sin(2*thetaprime)*2*Inp1_prev/arn
                # Inp1o3 = (2*W301*np.sin(thetaprime)+6*W303*np.sin(3*thetaprime))*(2*Inp1_prev)**(3/2)/arn
                # Inp1o4 = (4*W402*np.sin(2*thetaprime)+8*W404*np.sin(4*thetaprime))*(2*Inp1_prev)**(2)/arn
                Inp1_next = Iprime - Inp1o1 - Inp1o2 - Inp1o3 - Inp1o4
                
                phinp1o2 = (W200 + 2*W202*np.cos(2*thetaprime))*2/arn**3
                phinp1o3 = 3*2**(3/2)*(W301*np.cos(thetaprime)+W303*np.cos(3*thetaprime))*Inp1_prev**0.5/arn**4
                phinp1o4 = 8*(W400+2*W402*np.cos(2*thetaprime)+2*W404*np.cos(4*thetaprime))*Inp1_prev/arn**5
                # #Istar scale
                # phinp1o2 = (W200 + 2*W202*np.cos(2*thetaprime))*2/arn**3
                # phinp1o3 = 3*2**(3/2)*(W301*np.cos(thetaprime)+W303*np.cos(3*thetaprime))*Inp1_prev**0.5/arn**3
                # phinp1o4 = 8*(W400+2*W402*np.cos(2*thetaprime)+2*W404*np.cos(4*thetaprime))*Inp1_prev/arn**3
                phinp1_next = phiprime - (phinp1o2+phinp1o3+phinp1o4)
                if np.abs((Inp1_next-Inp1_prev)/Inp1_next) < 1e-12 and np.abs((phinp1_next-phinp1_prev)/phinp1_next) < 1e-12: 
                    #break when satisfied
                    break
                Inp1_prev = Inp1_next
                phinp1_prev = phinp1_next
        except RuntimeWarning as e:
            print("Inp1 candidate", Inp1_prev)
            print("Iprime", Iprime)
        Inp1 = Inp1_next
        phinp1 = phinp1_next
        Knp1 = Kn + Inp1 - In
        # Knp1 = Kn + (Inp1 - In)*arn**2 #Istar scale
        hnp1 = np.sqrt(2*Inp1)*np.cos(phinp1)
        knp1 = np.sqrt(2*Inp1)*np.sin(phinp1)
        lambdanp1 = lambdan + self._periodfn(Knp1)
        # if exact_period==True:
        #     lambdanp1 = lambdan + 2*np.pi*np.abs(1/((1+np.sqrt(8*Knp1/3)*epsilon)**(-3/2)-1)) #same lambda
        # else:
        #     lambdanp1 = lambdan + 4*np.pi/(3*epsilon*np.sqrt(8*Knp1/3))
        return Inp1, Knp1, phinp1, lambdanp1, hnp1, knp1

    def namouni_10o_step(self, In, Kn, phin, lambdan, arn, ind):
        """Tenth-order Namouni mapping step."""
        """The tenth order mapping involves an intermediate step and implicit scheme for eccentricity update."""
        """This calculates the implicit update iteratively instead of explicitly solving for the next value."""
        #define all coeffs in here so that we won't get confused
        #format WABC = W^{A,B}_C
        W200, W400, W600, W800, W1000 = -1,-3/8,-5/24,-35/256,-63/640
        W101, W301, W501, W701, W901 = 3.359350011, 0.1430675142, 0.2652528982, 0.1598760057, 0.1125398541
        W202, W402, W602, W802, W1002 = -2.678665722, 0.5232506456, -0.2880525893, -0.1146109541, -0.09264257446
        W303, W503, W703, W903 = 2.639498803, -1.270297850, 0.5079066956, 0.04027194541
        W404, W604, W804, W1004 = -2.855951527, 2.244981532, -1.030263325, 0.1370220235
        W505, W705, W905 = 3.259400253, -3.586274249, 2.034584646
        W606, W806, W1006 = -3.850815349, 5.467708834, -3.807558748
        W707, W907 = 4.661571101, -8.124434579
        W808, W1008 = -5.745863967, 11.88290720
        W909 = 7.181868977
        W10010 = -9.077037610
        hn = np.sqrt(2*In)*np.cos(phin)
        kn = np.sqrt(2*In)*np.sin(phin)
     
        if ind == 1:
            hprime = hn + 2*W101*np.sin(lambdan)/arn**2 # + for Namouni, - for pomega
            kprime = kn - 2*W101*np.cos(lambdan)/arn**2 # - for Namouni, + for pomega
            # hprime = hn + 2*W101*np.sin(lambdan)/arn
            # kprime = kn - 2*W101*np.cos(lambdan)/arn
        else: #from DQT, the sign switches for the outer pair
            hprime = hn - 2*W101*np.sin(lambdan)/arn**2
            kprime = kn + 2*W101*np.cos(lambdan)/arn**2 
            # hprime = hn - 2*W101*np.sin(lambdan)/arn
            # kprime = kn + 2*W101*np.cos(lambdan)/arn
        phiprime = np.arctan2(kprime, hprime) # 4 quadrant arctan
        #thetaprime = (pomegaprime - lambdan)
        thetaprime = (phiprime - lambdan)
    
        
        #implicit I update
        Iprime = 0.5*(hprime**2 + kprime**2)
        
        Inp1_prev = Iprime #first guess = Iprime
        phinp1_prev = phiprime
        try:
            while True: #iteratively update using implicit scheme
                Inp1o1 = 2*(W301/arn**4*(2*Inp1_prev)**1.5+W501/arn**6*(2*Inp1_prev)**2.5+W701/arn**8*(2*Inp1_prev)**3.5+W901/arn**10*(2*Inp1_prev)**4.5)*np.sin(thetaprime)
                Inp1o2 = 4*(W202/arn**3*(2*Inp1_prev)+W402/arn**5*(2*Inp1_prev)**2+W602/arn**7*(2*Inp1_prev)**3+W802/arn**9*(2*Inp1_prev)**4+W1002/arn**11*(2*Inp1_prev)**5)*np.sin(2*thetaprime)
                Inp1o3 = 6*(W303/arn**4*(2*Inp1_prev)**1.5+W503/arn**6*(2*Inp1_prev)**2.5+W703/arn**8*(2*Inp1_prev)**3.5+W903/arn**10*(2*Inp1_prev)**4.5)*np.sin(3*thetaprime)
                Inp1o4 = 8*(W404/arn**5*(2*Inp1_prev)**2+W604/arn**7*(2*Inp1_prev)**3+W804/arn**9*(2*Inp1_prev)**4+W1004/arn**11*(2*Inp1_prev)**5)*np.sin(4*thetaprime)
                Inp1o5 = 10*(W505/arn**6*(2*Inp1_prev)**2.5+W705/arn**8*(2*Inp1_prev)**3.5+W905/arn**10*(2*Inp1_prev)**4.5)*np.sin(5*thetaprime)
                Inp1o6 = 12*(W606/arn**7*(2*Inp1_prev)**3+W806/arn**9*(2*Inp1_prev)**4+W1006/arn**11*(2*Inp1_prev)**5)*np.sin(6*thetaprime)
                Inp1o7 = 14*(W707/arn**8*(2*Inp1_prev)**3.5+W907/arn**10*(2*Inp1_prev)**4.5)*np.sin(7*thetaprime)
                Inp1o8 = 16*(W808/arn**9*(2*Inp1_prev)**4+W1008/arn**11*(2*Inp1_prev)**5)*np.sin(8*thetaprime)
                Inp1o9 = 18*(W909/arn**10*(2*Inp1_prev)**4.5)*np.sin(9*thetaprime)
                Inp1o10 = 20*(W10010/arn**11*(2*Inp1_prev)**5)*np.sin(10*thetaprime)


                # Inp1o1 = 2*(W301*(2*Inp1_prev)**1.5 + W501*(2*Inp1_prev)**2.5 + W701*(2*Inp1_prev)**3.5 + W901*(2*Inp1_prev)**4.5)*np.sin(thetaprime)/arn
                # Inp1o2 = 4*(W202*(2*Inp1_prev) + W402*(2*Inp1_prev)**2 + W602*(2*Inp1_prev)**3 + W802*(2*Inp1_prev)**4 + W1002*(2*Inp1_prev)**5)*np.sin(2*thetaprime)/arn
                # Inp1o3 = 6*(W303*(2*Inp1_prev)**1.5 + W503*(2*Inp1_prev)**2.5 + W703*(2*Inp1_prev)**3.5 + W903*(2*Inp1_prev)**4.5)*np.sin(3*thetaprime)/arn
                # Inp1o4 = 8*(W404*(2*Inp1_prev)**2 + W604*(2*Inp1_prev)**3 + W804*(2*Inp1_prev)**4 + W1004*(2*Inp1_prev)**5)*np.sin(4*thetaprime)/arn
                # Inp1o5 = 10*(W505*(2*Inp1_prev)**2.5 + W705*(2*Inp1_prev)**3.5 + W905*(2*Inp1_prev)**4.5)*np.sin(5*thetaprime)/arn
                # Inp1o6 = 12*(W606*(2*Inp1_prev)**3 + W806*(2*Inp1_prev)**4 + W1006*(2*Inp1_prev)**5)*np.sin(6*thetaprime)/arn
                # Inp1o7 = 14*(W707*(2*Inp1_prev)**3.5 + W907*(2*Inp1_prev)**4.5)*np.sin(7*thetaprime)/arn
                # Inp1o8 = 16*(W808*(2*Inp1_prev)**4 + W1008*(2*Inp1_prev)**5)*np.sin(8*thetaprime)/arn
                # Inp1o9 = 18*(W909*(2*Inp1_prev)**4.5)*np.sin(9*thetaprime)/arn
                # Inp1o10 = 20*(W10010*(2*Inp1_prev)**5)*np.sin(10*thetaprime)/arn                
        
                Inp1_next = Iprime-Inp1o1-Inp1o2-Inp1o3-Inp1o4-Inp1o5-Inp1o6-Inp1o7-Inp1o8-Inp1o9-Inp1o10
                
                phinp1o0 = W200*2/arn**3+W400*2**2*2*Inp1_prev/arn**5+W600*2**3*3*Inp1_prev**2/arn**7+W800*2**4*4*Inp1_prev**3/arn**9+W1000*2**5*5*Inp1_prev**4/arn**11
                phinp1o1 = 2*(W301*2**1.5*1.5*Inp1_prev**0.5/arn**4+W501*2**2.5*2.5*Inp1_prev**1.5/arn**6+W701*2**3.5*3.5*Inp1_prev**2.5/arn**8+W901*2**4.5*4.5*Inp1_prev**3.5/arn**10)*np.cos(thetaprime)
                phinp1o2 = 2*(W202*2/arn**3+W402*2**2*2*Inp1_prev/arn**5+W602*2**3*3*Inp1_prev**2/arn**7+W802*2**4*4*Inp1_prev**3/arn**9+W1002*2**5*5*Inp1_prev**4/arn**11)*np.cos(2*thetaprime)
                phinp1o3 = 2*(W303*2**1.5*1.5*Inp1_prev**0.5/arn**4+W503*2**2.5*2.5*Inp1_prev**1.5/arn**6+W703*2**3.5*3.5*Inp1_prev**2.5/arn**8+W903*2**4.5*4.5*Inp1_prev**3.5/arn**10)*np.cos(3*thetaprime)
                phinp1o4 = 2*(W404*2**2*2*Inp1_prev/arn**5+W604*2**3*3*Inp1_prev**2/arn**7+W804*2**4*4*Inp1_prev**3/arn**9+W1004*2**5*5*Inp1_prev**4/arn**11)*np.cos(4*thetaprime)
                phinp1o5 = 2*(W505*2**2.5*2.5*Inp1_prev**1.5/arn**6+W705*2**3.5*3.5*Inp1_prev**2.5/arn**8+W905*2**4.5*4.5*Inp1_prev**3.5/arn**10)*np.cos(5*thetaprime)
                phinp1o6 = 2*(W606*2**3*3*Inp1_prev**2/arn**7+W806*2**4*4*Inp1_prev**3/arn**9+W1006*2**5*5*Inp1_prev**4/arn**11)*np.cos(6*thetaprime)
                phinp1o7 = 2*(W707*2**3.5*3.5*Inp1_prev**2.5/arn**8+W907*2**4.5*4.5*Inp1_prev**3.5/arn**10)*np.cos(7*thetaprime)
                phinp1o8 = 2*(W808*2**4*4*Inp1_prev**3/arn**9+W1008*2**5*5*Inp1_prev**4/arn**11)*np.cos(8*thetaprime)
                phinp1o9 = 2*(W909*2**4.5*4.5*Inp1_prev**3.5/arn**10)*np.cos(9*thetaprime)
                phinp1o10 = 2*(W10010*2**5*5*Inp1_prev**4/arn**11)*np.cos(10*thetaprime)

                # phinp1o0 = (W200*2 + W400*2**2*2*Inp1_prev + W600*2**3*3*Inp1_prev**2 + W800*2**4*4*Inp1_prev**3 + W1000*2**5*5*Inp1_prev**4)/arn**3
                # phinp1o1 = 2*(W301*2**1.5*1.5*Inp1_prev**0.5 + W501*2**2.5*2.5*Inp1_prev**1.5 + W701*2**3.5*3.5*Inp1_prev**2.5 + W901*2**4.5*4.5*Inp1_prev**3.5)*np.cos(thetaprime)/arn**3
                # phinp1o2 = 2*(W202*2 + W402*2**2*2*Inp1_prev + W602*2**3*3*Inp1_prev**2 + W802*2**4*4*Inp1_prev**3 + W1002*2**5*5*Inp1_prev**4)*np.cos(2*thetaprime)/arn**3
                # phinp1o3 = 2*(W303*2**1.5*1.5*Inp1_prev**0.5 + W503*2**2.5*2.5*Inp1_prev**1.5 + W703*2**3.5*3.5*Inp1_prev**2.5 + W903*2**4.5*4.5*Inp1_prev**3.5)*np.cos(3*thetaprime)/arn**3
                # phinp1o4 = 2*(W404*2**2*2*Inp1_prev + W604*2**3*3*Inp1_prev**2 + W804*2**4*4*Inp1_prev**3 + W1004*2**5*5*Inp1_prev**4)*np.cos(4*thetaprime)/arn**3
                # phinp1o5 = 2*(W505*2**2.5*2.5*Inp1_prev**1.5 + W705*2**3.5*3.5*Inp1_prev**2.5 + W905*2**4.5*4.5*Inp1_prev**3.5)*np.cos(5*thetaprime)/arn**3
                # phinp1o6 = 2*(W606*2**3*3*Inp1_prev**2 + W806*2**4*4*Inp1_prev**3 + W1006*2**5*5*Inp1_prev**4)*np.cos(6*thetaprime)/arn**3
                # phinp1o7 = 2*(W707*2**3.5*3.5*Inp1_prev**2.5 + W907*2**4.5*4.5*Inp1_prev**3.5)*np.cos(7*thetaprime)/arn**3
                # phinp1o8 = 2*(W808*2**4*4*Inp1_prev**3 + W1008*2**5*5*Inp1_prev**4)*np.cos(8*thetaprime)/arn**3
                # phinp1o9 = 2*(W909*2**4.5*4.5*Inp1_prev**3.5)*np.cos(9*thetaprime)/arn**3
                # phinp1o10 = 2*(W10010*2**5*5*Inp1_prev**4)*np.cos(10*thetaprime)/arn**3
                phinp1_next = phiprime-phinp1o0-phinp1o1-phinp1o2-phinp1o3-phinp1o4-phinp1o5-phinp1o6-phinp1o7-phinp1o8-phinp1o9-phinp1o10
                if np.abs((Inp1_next-Inp1_prev)/Inp1_next) < 1e-12 and np.abs((phinp1_next-phinp1_prev)/phinp1_next) < 1e-12: 
                    #break when satisfied
                    break
                Inp1_prev = Inp1_next
                phinp1_prev = phinp1_next
    
        except RuntimeWarning as e:
            print("Inp1 candidate", Inp1_prev)
            print("Iprime", Iprime)
                
            
            
        Inp1 = Inp1_next
        phinp1 = phinp1_next
        Knp1 = Kn + Inp1 - In
        # Knp1 = Kn + (Inp1 - In)*arn**2 #Istar scale
        hnp1 = np.sqrt(2*Inp1)*np.cos(phinp1)
        knp1 = np.sqrt(2*Inp1)*np.sin(phinp1)
        lambdanp1 = lambdan + self._periodfn(Knp1)
        # if exact_period==True:
        #     lambdanp1 = lambdan + 2*np.pi*np.abs(1/((1+np.sqrt(8*Knp1/3)*epsilon)**(-3/2)-1)) #same lambda
        # else:
        #     lambdanp1 = lambdan + 4*np.pi/(3*epsilon*np.sqrt(8*Knp1/3))
        return Inp1, Knp1, phinp1, lambdanp1, hnp1, knp1

        #This function runs Namouni mapping for multiple conjunctions
    def runmap(self, I0, K0, phi0, lambda0, Nconj, order):
        try:                # minimize function later passes a 1-element list, so make it work for both scalar and list
            K0 = K0[0]
        except:
            pass
        Is, Ks, phis, lambdas, hs, ks = np.zeros(Nconj+1), np.zeros(Nconj+1), np.zeros(Nconj+1), np.zeros(Nconj+1), np.zeros(Nconj+1), np.zeros(Nconj+1)
        Is[0] = I0
        Ks[0] = K0
        phis[0] = phi0
        lambdas[0] = lambda0
        ar0 = np.sqrt(8*K0/3)
        for i in range(Nconj):
            if order==1:
                Is[i+1], Ks[i+1], phis[i+1], lambdas[i+1], hs[i+1], ks[i+1] = self.namouni_fo_step(Is[i], Ks[i], phis[i], lambdas[i], ar0, self.ind)
            elif order==2:
                Is[i+1], Ks[i+1], phis[i+1], lambdas[i+1], hs[i+1], ks[i+1] = self.namouni_so_step(Is[i], Ks[i], phis[i], lambdas[i], ar0, self.ind)
            elif order==4:
                Is[i+1], Ks[i+1], phis[i+1], lambdas[i+1], hs[i+1], ks[i+1] = self.namouni_4o_step(Is[i], Ks[i], phis[i], lambdas[i], ar0, self.ind)
            elif order==10:
                Is[i+1], Ks[i+1], phis[i+1], lambdas[i+1], hs[i+1], ks[i+1] = self.namouni_10o_step(Is[i], Ks[i], phis[i], lambdas[i], ar0, self.ind)

        return Is, Ks, phis, lambdas, hs, ks
    
    def __call__(self,X,Nconj,order):
        epsilon = self.epsilon
        e0, a_r0, phi0, lambda0 = X
        I0 = (e0/epsilon)**2/2
        K0 = 3/8*a_r0**2
    
        Is, Ks, phis, lambdas, hs, ks = self.runmap(I0, K0, phi0, lambda0, Nconj, order)
        return Is, Ks, phis, lambdas, hs, ks

## Parameters
These match the 10:9 panel of the four-section plot in `testmap.ipynb`. For the 3:2 case, set `J = 3`.

In [3]:
epsilon = (1e-6/3)**(1/3)   # Hill radius scale
mutot = 3*epsilon**3        # planet-star mass ratio
J = 10                      # J:J-1 resonance sets the centre of the a_r range
ind = 1                     # 1: massive planet inside the test particle (the only case handled below)
order = 10                  # Namouni map order (1, 2, 4 or 10)

a1 = 1
a2 = (J/(J-1))**(2/3)*a1
ecross = (a2-a1)/a1
ar_res = ecross/epsilon     # a_r at exact resonance, in Hill units
ecc = 3*epsilon             # eccentricity at the bottom of the a_r range

# a_r window of the section, and the conserved quantity C = K - I that sets the
# eccentricity for each a_r (I_list = K0_list - C in testmap)
x0_arr = np.linspace(0, 0.25, 40)
ar0_list = (1+x0_arr)*ar_res
C = 3/8*ar0_list[0]**2 - (ecc/epsilon)**2/2

Nconj = 200                 # conjunctions per clicked orbit on the section (the animation extends the map as it goes)
Ntrace = 500                # conjunctions whose passes are traced in the zoom panel (more slows wide zooms)
samples_per_orbit = 1000    # trail positions per planet orbit in the animation (fine enough for the zoom panel)
zoom_half = 30              # initial half-width of the zoom box, in Hill radii
zoom_min, zoom_max = 0.1, 100   # range of the zoom slider, in Hill radii
zoom_angle = 0.0            # initial angle of the zoom box centre in the rotating frame (the pericentre is at pi/2)
Nlines = 20                 # conjunction lines kept on screen
conj_step = 0.02            # time step for finding conjunctions, in planet orbits
speed_max = 100             # top of the speed slider, in planet orbits per second
nbody_samples = 50          # N-body samples per planet orbit (positions in between are Kepler-propagated)
nbody_chunk = 5             # planet orbits integrated at a time, ahead of the animation

font_size = 16              # base font size of the window (titles and labels are a little larger)
fig_size = (16, 8.4)        # window size in inches
print(f"a_r range: {ar0_list[0]:.2f} - {ar0_list[-1]:.2f} Hill units")

a_r range: 10.49 - 13.12 Hill units


## Orbits from a clicked point
The left panel's coordinates are converted to the map's: $\theta = -$(conjunction angle) and $a_r = ((P_{\rm test}/P_{\rm planet})^{2/3} - 1)/\epsilon$.

**Namouni map (left panel).** `MapOrbit` runs the map for `Nconj` conjunctions from $\lambda_0 = 0$, $\phi_0 = \theta_0$, and gives the section points.

**N-body (right panels).** `NBodyOrbit` integrates the star (mass 1), the planet (mass ratio `mutot`, circular orbit at $a_p = 1$ AU) and a massless test body with REBOUND's IAS15. Time is measured in planet orbits. The test body starts with $a = a_p(1 + \epsilon a_r)$, $e$ from $C = K - I$ and $\varpi = \theta_0 + \pi$ (the relation `threebody_interactive.ipynb` uses, $\theta = \varpi - \lambda_{conj} + \pi$), half a synodic period before the first conjunction, with both mean longitudes reaching 0 there. So it has the clicked elements while it is far from the planet, and the first conjunction is at the clicked $\theta_0$.

The integration is sampled `nbody_samples` times per planet orbit, and positions in between come from Kepler-propagating each body's heliocentric osculating elements from the sample before; over one sample interval the planet's pull is far too small to see. Only the last few planet orbits of samples are kept, so it can run indefinitely.

The rotating frame turns each position by $\pi/2 - \varpi$, with the test body's osculating $\varpi$ at that time, so the pericentre is always at the top.

The conjunction lines mark when the planet and the test body are in the same direction from the star, i.e. when their *true* longitudes are equal, found from the N-body positions. These also divide the motion into segments; the zoom panel's traces of earlier passes use the test body's osculating $a$ and $e$ one orbit after each conjunction.

In [4]:
def eccentricity_for(ar0):
    """Eccentricity on the section's C = K - I surface, or None if a_r is below it."""
    I0 = 3/8*ar0**2 - C
    return np.sqrt(2*I0)*epsilon if I0 > 0 else None


# The left panel shows the conjunction angle (-theta) and the period ratio instead of theta and a_r
def ratio_from_ar(ar):
    """Period ratio P_test/P_planet for a relative semimajor axis a_r (Hill units)."""
    return (1 + epsilon*np.asarray(ar))**1.5

def ar_from_ratio(ratio):
    return (np.asarray(ratio)**(2/3) - 1)/epsilon

def conj_angle(theta):
    """Conjunction angle -theta, wrapped into [-pi, pi)."""
    return (np.pi - np.asarray(theta)) % (2*np.pi) - np.pi


def solve_kepler(M, e, tol=1e-12):
    """Eccentric anomaly E from mean anomaly M (Newton iteration)."""
    E = M + e*np.sin(M)
    for _ in range(50):
        dE = (E - e*np.sin(E) - M)/(1 - e*np.cos(E))
        E -= dE
        if np.max(np.abs(dE)) < tol:
            break
    return E


class MapOrbit:
    """The Namouni map's orbit from a clicked (theta0, ar0): the points it adds to the surface of section."""

    def __init__(self, theta0, ar0, nconj=Nconj):
        n_map = NamouniMap(mutot, ind, order)
        # theta = phi - lambda, so start at lambda0 = 0, phi0 = theta0
        Is, Ks, phis, lams, _, _ = n_map((eccentricity_for(ar0), ar0, theta0, 0.0), nconj, order)
        self.theta = (phis - lams + np.pi) % (2*np.pi) - np.pi
        self.ar = np.sqrt(8/3*Ks)


class NBodyOrbit:
    """
    REBOUND integration (IAS15) of the star, the planet and a massless test particle, started from a
    clicked (theta0, ar0) half a synodic period before the first conjunction, so the particle has
    the clicked elements while it is far from the planet. Integrated a chunk at a time whenever the
    animation needs more, so it runs for as long as it's left on; only the last few planet orbits of
    samples are kept.

    Time t is in planet orbits. Positions between samples come from Kepler-propagating each body's
    heliocentric osculating elements at the sample before (the planet's pull over one sample interval
    is far too small to see).
    """

    def __init__(self, theta0, ar0):
        a = a1*(1 + epsilon*ar0)
        n_q = (a/a1)**-1.5
        self.t_first = 0.5/(1 - n_q)   # time of the first conjunction, in planet orbits
        sim = rebound.Simulation()
        sim.integrator = "ias15"
        sim.add(m=1)
        sim.add(m=mutot, a=a1, l=-2*np.pi*self.t_first)
        # both mean longitudes reach 0 at t_first; theta = pomega - lambda_conj - pi there
        sim.add(primary=sim.particles[0], a=a, e=eccentricity_for(ar0), pomega=theta0 + np.pi,
                l=-2*np.pi*n_q*self.t_first)
        sim.N_active = 2
        sim.move_to_com()
        self.sim = sim
        self.mu = np.array([1 + mutot, 1.0])   # G(M_star + m) for the planet and the particle
        self._xyz, self._vel = np.zeros((3, 3)), np.zeros((3, 3))
        self.ts = np.zeros(0)
        self.xy = np.zeros((0, 2, 2))    # heliocentric (x, y) of planet and particle at the samples
        self.el = np.zeros((0, 5, 2))    # their elements: a, e, pomega, M, n (rad per planet orbit)
        self.bounds = [0.0]              # segment boundaries: start, then each conjunction
        self.t_done = []                 # when the particle has gone once round after each boundary
        self.seg_ae = []                 # the particle's (a, e) at those times
        self._samples(*self._state())
        self._advance(nbody_chunk)

    def _state(self):
        self.sim.serialize_particle_data(xyz=self._xyz, vxvyvz=self._vel)
        return (self.sim.t/(2*np.pi), self._xyz[1:, :2] - self._xyz[0, :2],
                self._vel[1:, :2] - self._vel[0, :2])

    def _elements(self, xy, v):
        """Heliocentric planar osculating elements, vectorised over samples and bodies."""
        mu = self.mu
        r = np.hypot(xy[..., 0], xy[..., 1])
        v2 = (v**2).sum(-1)
        rv = (xy*v).sum(-1)
        a = 1/(2/r - v2/mu)
        ev = ((v2 - mu/r)[..., None]*xy - rv[..., None]*v)/mu[..., None]
        e = np.hypot(ev[..., 0], ev[..., 1])
        w = np.arctan2(ev[..., 1], ev[..., 0])
        f = np.arctan2(xy[..., 1], xy[..., 0]) - w
        E = np.arctan2(np.sqrt(1 - e**2)*np.sin(f), e + np.cos(f))
        n = 2*np.pi*np.sqrt(mu/a**3)
        return np.stack([a, e, w, E - e*np.sin(E), n], axis=1)

    def _samples(self, t, xy, v):
        t, xy, v = np.atleast_1d(t), np.asarray(xy).reshape(-1, 2, 2), np.asarray(v).reshape(-1, 2, 2)
        self.ts = np.concatenate([self.ts, t])
        self.xy = np.concatenate([self.xy, xy])
        self.el = np.concatenate([self.el, self._elements(xy, v)])

    def _advance(self, t_end, t_keep=None):
        """Integrate to t_end, find the conjunctions on the way, and drop samples before t_keep."""
        n0 = len(self.ts)
        T, X, V = [], [], []
        dt = 2*np.pi/nbody_samples
        while self.sim.t < 2*np.pi*t_end:
            # stop at the first step past the next sample time; Kepler propagation fills in between
            self.sim.integrate(self.sim.t + dt, exact_finish_time=0)
            t, xy, v = self._state()
            T.append(t); X.append(xy.copy()); V.append(v.copy())
        self._samples(T, X, V)
        # conjunctions: the true longitude gap changes sign (except at opposition, where it wraps)
        p, q = self.xy[n0 - 1:, 0], self.xy[n0 - 1:, 1]
        gap = np.arctan2(p[:, 0]*q[:, 1] - p[:, 1]*q[:, 0], (p*q).sum(-1))
        hit = np.flatnonzero((gap[:-1]*gap[1:] < 0) & (np.abs(gap[:-1]) < np.pi/2) & (np.abs(gap[1:]) < np.pi/2))
        ts = self.ts[n0 - 1:]
        for i in hit:
            lo, hi, g_lo = ts[i], ts[i + 1], gap[i]
            for _ in range(40):   # bisection
                mid = (lo + hi)/2
                if self.longitude_gap(np.array([mid]))[0]*g_lo > 0:
                    lo = mid
                else:
                    hi = mid
            self.bounds.append(hi)
        # the particle's (a, e) one orbit after each boundary, for the traces in the zoom panel
        while len(self.t_done) < len(self.bounds):
            b = self.bounds[len(self.t_done)]
            k = self._index(b)
            t_e = b + (self.el[k, 0, 1]/a1)**1.5
            if t_e > self.ts[-1]:
                break
            k = self._index(t_e)
            self.t_done.append(t_e)
            self.seg_ae.append((self.el[k, 0, 1], self.el[k, 1, 1]))
        if t_keep is not None:
            keep = max(0, np.searchsorted(self.ts, t_keep) - 1)
            self.ts, self.xy, self.el = self.ts[keep:], self.xy[keep:], self.el[keep:]

    def _index(self, t):
        return np.clip(np.searchsorted(self.ts, t, side='right') - 1, 0, len(self.ts) - 1)

    def ensure(self, t):
        """Make sure the integration has reached the latest of times t."""
        t = np.atleast_1d(t)
        if self.ts[-1] < t.max():
            self._advance(t.max() + nbody_chunk, t_keep=t.min() - 2)

    def positions(self, t):
        """Heliocentric (x, y) of the planet and the test particle at times t (planet orbits), shape (len(t), 2, 2)."""
        t = np.atleast_1d(np.asarray(t, dtype=float))
        self.ensure(t)
        k = self._index(t)
        a, e, w, M0, n = (self.el[k, j] for j in range(5))
        E = solve_kepler(M0 + n*(t - self.ts[k])[:, None], e)
        xo, yo = a*(np.cos(E) - e), a*np.sqrt(1 - e**2)*np.sin(E)   # in the orbital plane, pericentre on x
        return np.stack([xo*np.cos(w) - yo*np.sin(w), xo*np.sin(w) + yo*np.cos(w)], axis=2)

    def pomega(self, t):
        """The test particle's osculating pericentre longitude at times t."""
        return self.el[self._index(t), 2, 1]

    def frame_positions(self, t):
        """Like positions, but in the frame that rotates with the particle's pericentre, which is at the top."""
        t = np.atleast_1d(np.asarray(t, dtype=float))
        pos = self.positions(t)
        psi = np.pi/2 - self.pomega(t)
        c, s = np.cos(psi)[:, None], np.sin(psi)[:, None]
        return np.stack([c*pos[:, :, 0] - s*pos[:, :, 1], s*pos[:, :, 0] + c*pos[:, :, 1]], axis=2)

    def longitude_gap(self, t):
        """True longitude of the particle minus that of the planet at times t, in (-pi, pi]."""
        pos = self.positions(t)
        (xp, yp), (xq, yq) = pos[:, 0].T, pos[:, 1].T
        return np.arctan2(xp*yq - yp*xq, xp*xq + yp*yq)

    def segment(self, t):
        """Number of conjunctions before time t."""
        self.ensure(t)
        return np.searchsorted(self.bounds, t, side='right') - 1

    def conjunctions(self, t0, t1):
        """Times between t0 and t1 when the planet and the particle are in the same direction from the star."""
        self.ensure(t1)
        b = np.array(self.bounds[1:])
        return b[(b > t0) & (b <= t1)]

    def conjunction_angles(self, t):
        """Direction of the conjunctions at times t, as angles in the rotating frame."""
        pos = self.positions(t)
        return np.arctan2(pos[:, 0, 1], pos[:, 0, 0]) + np.pi/2 - self.pomega(t)

    def segments_done(self, t):
        """Number of segments (between conjunctions) the particle has been all the way round in by time t."""
        self.ensure(t)
        return int(np.searchsorted(self.t_done, t, side='right'))

    def trace_elements(self, lo, hi):
        """(a, e) of the particle in segments lo to hi - 1, as column arrays."""
        ae = np.array(self.seg_ae[lo:hi]).reshape(-1, 2)
        return ae[:, :1], ae[:, 1:]

## Interactive window

In [ ]:
%matplotlib tk
assert 'NBodyOrbit' in globals(), "Run the cells above first (Kernel > Restart Kernel and Run All Cells)."
plt.close('all')   # re-running this cell replaces the window

# large text, for showing on a screen at an exhibition
plt.rcParams.update({"font.size": font_size, "axes.titlesize": font_size + 2, "axes.labelsize": font_size + 1,
                     "xtick.labelsize": font_size - 1, "ytick.labelsize": font_size - 1})

# The figure opens in its own window. Unlike the browser backend, the Tk backend can blit: each
# frame redraws only the moving artists over a saved copy of everything else.
fig = plt.figure(figsize=fig_size)
fig.canvas.manager.set_window_title("Three-body problem: Namouni map and N-body")
gs = fig.add_gridspec(1, 3, left=0.07, right=0.985, top=0.88, bottom=0.27, wspace=0.38)
ax_sec, ax_xy, ax_zoom = (fig.add_subplot(gs[0, k]) for k in range(3))
for ax in (ax_sec, ax_xy, ax_zoom):
    ax.set_box_aspect(1)   # three equal square panels

# left: surface of section from the Namouni map (empty until the first click)
ax_sec.set_title("Namouni map\n(click to start an orbit)")
ax_sec.set_xlabel("Conjunction angle")
ax_sec.set_ylabel(r"Period ratio  $P_{\rm test}\,/\,P_{\rm planet}$")
ax_sec.set_xlim(-np.pi, np.pi)
ax_sec.set_ylim(ratio_from_ar(ar0_list[0]), ratio_from_ar(ar0_list[-1]))
ax_sec.xaxis.set_major_locator(MultipleLocator(np.pi/2))
ax_sec.xaxis.set_major_formatter(FuncFormatter(
    lambda v, _: {-2: r"$-\pi$", -1: r"$-\pi/2$", 0: "0", 1: r"$\pi/2$", 2: r"$\pi$"}.get(round(2*v/np.pi), "")))
ax_sec.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.3f}"))


def make_sky(seed, half, n=1200, m=4000):
    """
    A star field covering -half..half AU in x and y, in the rotating frame's coordinates, so
    the middle and zoom panels show the same sky: faint nebulae as an image, and stars as points
    (which stay sharp however far the zoom panel magnifies).
    """
    rng = np.random.default_rng(seed)
    img = np.empty((n, n, 3))
    img[:] = (0.008, 0.012, 0.035)
    # faint nebulae: smoothed noise, thresholded
    k = np.hypot(np.fft.fftfreq(n)[:, None], np.fft.fftfreq(n)[None, :])
    for colour, scale, amp in (((0.35, 0.12, 0.55), 0.004, 0.14), ((0.10, 0.28, 0.55), 0.0075, 0.12)):
        cloud = np.real(np.fft.ifft2(np.fft.fft2(rng.normal(size=(n, n)))*np.exp(-(k/scale)**2)))
        cloud = np.clip((cloud - cloud.mean())/cloud.std() - 0.4, 0, None)**1.5*amp
        img += cloud[..., None]*np.array(colour)
    # stars: positions in AU, brightness and a slight tint
    tints = np.array([(1, 1, 1), (0.75, 0.85, 1), (1, 0.9, 0.7), (1, 0.75, 0.6)])
    stars = {"x": rng.uniform(-half, half, m), "y": rng.uniform(-half, half, m),
             "bright": np.clip(0.15 + 0.3*rng.pareto(2.2, m), 0, 1),
             "tint": tints[rng.choice(4, m, p=[0.55, 0.25, 0.15, 0.05])]}
    return {"img": np.clip(img, 0, 1), "half": half, "stars": stars}

def draw_sky(ax, sky):
    """Draw the sky behind everything else in ax, in data coordinates."""
    h, st = sky["half"], sky["stars"]
    ax.set_facecolor(sky["img"][0, 0])
    ax.imshow(sky["img"], origin='lower', extent=(-h, h, -h, h), interpolation='bilinear', zorder=-10)
    b = st["bright"]
    ax.scatter(st["x"], st["y"], s=0.4 + 3.5*b**2, c=np.column_stack([st["tint"], b]), linewidths=0, zorder=-9)
    glow = np.argsort(b)[-40:]   # a soft halo round the brightest
    ax.scatter(st["x"][glow], st["y"][glow], s=60*b[glow], c=np.column_stack([st["tint"][glow], 0.12*b[glow]]),
               linewidths=0, zorder=-9)

# colours that show up on the space background
fg = '0.9'            # planet, its trails and the zoom box
zoom_color = fg

# middle: frame rotating with the pericentre, with the zoom box (click or drag to move it)
ax_xy.set_title("N-body, test body's pericentre up\n(click or drag to move the box)")
ax_xy.set_aspect('equal')
ax_xy.set_xlabel("x (AU)")
ax_xy.set_ylabel("y (AU)")
lim = 1.15*a2*(1 + 0.25*ecross)
ax_xy.set_xlim(-lim, lim)
ax_xy.set_ylim(-lim, lim)
sky = make_sky(1, 1.5*lim)   # a little beyond the panel, for a zoom box near its edge
draw_sky(ax_xy, sky)
ax_xy.plot(0, 0, '*', color='gold', markersize=18)
zoom_r = a1*(1 + epsilon*ar_res/2)   # midway between the planet's and the test particle's orbits
hill = epsilon*a1                    # one Hill radius, in AU (a_p = 1 AU)
zoom = {"centre": np.array([zoom_r*np.cos(zoom_angle), zoom_r*np.sin(zoom_angle)]),
        "half": zoom_half*hill}      # zoom box centre and half-width, in AU
zoom_box = Rectangle((0, 0), 1, 1, fill=False, ec=zoom_color, lw=1.5, ls='--')
ax_xy.add_patch(zoom_box)
zoom_marker, = ax_xy.plot([], [], '+', color=zoom_color, markersize=14, mew=1.5)   # marks the box when it is too small to see

# right: zoom on the dashed box, in the same rotating frame and units
ax_zoom.set_title("Zoom on the dashed box\n(drag to pan, scroll to zoom)")
ax_zoom.set_aspect('equal')
ax_zoom.set_xlabel("x (AU)")
ax_zoom.set_ylabel("y (AU)")
draw_sky(ax_zoom, sky)   # the same sky, so the zoom shows what is in the box
_phi = np.linspace(0, 2*np.pi, 4001)
ax_zoom.plot(a1*np.cos(_phi), a1*np.sin(_phi), ':', color='0.6', lw=1)   # the planet's circular orbit
ax_zoom.plot(0, 0, '*', color='gold', markersize=18)
ax_zoom.grid(True, color='0.6', alpha=0.3)
for spine in ax_zoom.spines.values():
    spine.set_edgecolor('k')
zoom_history, = ax_zoom.plot([], [], '-', lw=0.8, alpha=0.45)   # earlier passes, zoom panel only

# ======================
# Controls along the bottom
# ======================
interval = 30   # ms between animation frames
speed_slider = Slider(fig.add_axes([0.15, 0.135, 0.25, 0.04]), "Speed", -1.5, np.log10(speed_max), valinit=np.log10(0.5))
zoom_slider = Slider(fig.add_axes([0.15, 0.06, 0.25, 0.04]), "Zoom half-width",
                     np.log10(zoom_min), np.log10(zoom_max), valinit=np.log10(zoom_half))

def zoom_text(val):
    h = 10**val*hill
    return f"{h:.2g} AU" if h >= 0.01 else f"{h:.1e} AU"

speed_slider.valtext.set_text(f"{10**speed_slider.val:.3g} planet orbits/s")   # the sliders run in log10
zoom_slider.valtext.set_text(zoom_text(zoom_slider.val))
pause_btn = Button(fig.add_axes([0.53, 0.065, 0.08, 0.08]), "Pause")
clear_btn = Button(fig.add_axes([0.62, 0.065, 0.08, 0.08]), "Clear")
ax_status = fig.add_axes([0.72, 0.03, 0.27, 0.15])
ax_status.axis('off')
status = ax_status.text(0, 0.5, "", va='center', fontsize=font_size - 3, family='monospace', wrap=True, animated=True)

def set_status(text, color='0.25'):
    status.set_text(text)
    status.set_color(color)

# ======================
# Animation state and blitting
# ======================
# The moving artists are animated: full redraws skip them, and each frame draws them over the saved
# background. Anything else that changes (section points, box, zoom, traces) triggers a full redraw,
# after which the background is saved again.
planet_dot, = ax_xy.plot([], [], 'o', color=fg, markersize=8, animated=True)
particle_dot, = ax_xy.plot([], [], 'o', markersize=7, animated=True)
planet_trail, = ax_xy.plot([], [], '-', color=fg, lw=1, alpha=0.5, animated=True)
particle_trail, = ax_xy.plot([], [], '-', lw=1.2, animated=True)
zoom_planet_trail, = ax_zoom.plot([], [], '-', color=fg, lw=1, alpha=0.5, animated=True)
zoom_planet_dot, = ax_zoom.plot([], [], 'o', color=fg, markersize=8, animated=True)
zoom_particle_trail, = ax_zoom.plot([], [], '-', lw=1.6, animated=True)
zoom_particle_dot, = ax_zoom.plot([], [], 'o', markersize=9, markeredgecolor='w', animated=True)
# conjunction lines, in both panels: the previous Nlines - 1 (dim) and the latest (bright)
old_lines = [ax.plot([], [], '-', color='0.45', lw=0.9, animated=True)[0] for ax in (ax_xy, ax_zoom)]
new_lines = [ax.plot([], [], '-', color='0.85', lw=1.3, animated=True)[0] for ax in (ax_xy, ax_zoom)]
moving = old_lines + new_lines + [planet_dot, particle_dot, planet_trail, particle_trail,
          zoom_planet_trail, zoom_planet_dot, zoom_particle_trail, zoom_particle_dot, status]
coloured = [particle_dot, particle_trail, zoom_particle_trail, zoom_particle_dot, zoom_history]
# orbit colours that show up both on the white section and on the space background
palette = ['#1f9bff', '#ff4f4f', '#2ecc40', '#ff9f1c', '#c77dff', '#00c2c7', '#ff5cad', '#c9b400']

orbits = []     # section artists for each clicked orbit
# orb: the orbit being animated; t: animation time in planet orbits; traces: what the traces were drawn for;
# traced: when they were last redrawn; conj: angles of the conjunction lines on screen, oldest first
anim = {"orb": None, "t": 0.0, "last": time.perf_counter(), "paused": False, "traces": None, "shown": 0.0,
        "traced": 0.0, "conj": []}
background = {"img": None}
trail_steps = np.linspace(-1, 0, samples_per_orbit + 1)   # the last planet orbit

def on_draw(event):
    """After every full redraw, save the background and draw the moving artists over it."""
    background["img"] = fig.canvas.copy_from_bbox(fig.bbox)
    for art in moving:
        fig.draw_artist(art)

def blit_frame():
    if background["img"] is None:
        return
    fig.canvas.restore_region(background["img"])
    for art in moving:
        fig.draw_artist(art)
    for ax in (ax_xy, ax_zoom, ax_status):
        fig.canvas.blit(ax.bbox)

def report_errors(fn):
    """Show errors from window callbacks in the status line and the notebook, and stop the animation."""
    @wraps(fn)
    def wrapper(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as err:
            anim["orb"] = None
            set_status(f"Stopped by an error in {fn.__name__}:\n{type(err).__name__}: {err}", 'crimson')
            traceback.print_exc()
            fig.canvas.draw_idle()
    return wrapper

def traces_in_box(orb, done):
    """
    Faint traces of the earlier passes through the zoom box. Between two conjunctions the test
    particle goes round nearly the same Kepler ellipse several times, so draw the part of that
    ellipse near the box (with the N-body elements one orbit after the conjunction), for the
    last Ntrace segments. In the rotating frame every one of these ellipses has its pericentre at the top.
    """
    a, e = orb.trace_elements(max(0, done - Ntrace), done)
    if len(a) == 0:
        return [], []
    (cx, cy), h = zoom["centre"], zoom["half"]
    rc, psi = np.hypot(cx, cy), np.arctan2(cy, cx)
    reach = 1.5*np.sqrt(2)*h   # beyond the box corners
    window = np.pi if rc <= reach else min(np.pi, 1.2*np.arcsin(reach/rc))
    # longitude step that keeps the straight segments within ~0.1 pixel of the ellipse
    dth = np.sqrt(0.0064*h/a1)
    th = psi + np.linspace(-window, window, int(np.clip(np.ceil(2*window/dth), 16, 1000)))
    r = a*(1 - e**2)/(1 + e*np.cos(th - np.pi/2))
    x, y = r*np.cos(th), r*np.sin(th)
    inside = np.any((np.abs(x - cx) < h) & (np.abs(y - cy) < h), axis=1)
    if not inside.any():
        return [], []
    x, y = x[inside], y[inside]
    gap = np.full((len(x), 1), np.nan)   # NaN after each arc, so the arcs aren't joined
    return np.hstack([x, gap]).ravel(), np.hstack([y, gap]).ravel()

def refresh_traces(force=False):
    """Redraw the traces when a segment finishes or the box changes; returns True if they changed."""
    orb = anim["orb"]
    if orb is None:
        return False
    done = orb.segments_done(anim["t"])
    key = (done, tuple(zoom["centre"]), zoom["half"])
    if not force and key == anim["traces"]:
        return False
    zoom_history.set_data(*traces_in_box(orb, done))
    anim["traces"] = key
    return True

def set_zoom(centre=None, half=None):
    """Move and/or resize the zoom box, and update the zoom panel to match."""
    if centre is not None:
        zoom["centre"] = np.asarray(centre, dtype=float)
    if half is not None:
        zoom["half"] = half
    (cx, cy), h = zoom["centre"], zoom["half"]
    zoom_box.set_bounds(cx - h, cy - h, 2*h, 2*h)
    zoom_marker.set_data([cx], [cy])
    ax_zoom.set_xlim(cx - h, cx + h)
    ax_zoom.set_ylim(cy - h, cy + h)
    # about 4 ticks at round numbers of AU, with just enough decimals
    raw = 2*h/4
    p = 10**np.floor(np.log10(raw))
    step = next(m*p for m in (1, 2, 5, 10) if m*p >= raw)
    decimals = max(0, -int(np.floor(np.log10(step) + 1e-9)))
    for axis in (ax_zoom.xaxis, ax_zoom.yaxis):
        axis.set_major_locator(MultipleLocator(step))
        axis.set_major_formatter(FuncFormatter(lambda v, _, d=decimals: f"{v:.{d}f}"))
    refresh_traces(force=True)
    fig.canvas.draw_idle()

def set_conj_lines(angles):
    """Draw the conjunction lines as rays from the star, past the edge of the panel."""
    for lines, a in ((old_lines, angles[:-1]), (new_lines, angles[-1:])):
        a = np.asarray(a, dtype=float)
        x = np.column_stack([0*a, 2*lim*np.cos(a), np.nan*a]).ravel()
        y = np.column_stack([0*a, 2*lim*np.sin(a), np.nan*a]).ravel()
        for line in lines:
            line.set_data(x, y)

def tick():
    """One animation frame: advance by the wall-clock time since the last one, at the slider's speed."""
    now = time.perf_counter()
    dt = min(now - anim["last"], 0.5)   # cap the jump after a long stall
    anim["last"] = now
    orb = anim["orb"]
    if orb is None or anim["paused"]:
        return
    t_prev = anim["t"]
    anim["t"] += 10**speed_slider.val*dt
    times = np.maximum(anim["t"] + trail_steps, 0)
    xy = orb.frame_positions(times)   # integrates as far as needed
    t_conj = orb.conjunctions(t_prev, anim["t"])
    if len(t_conj):
        anim["conj"] = (anim["conj"] + list(orb.conjunction_angles(t_conj)))[-Nlines:]
        set_conj_lines(anim["conj"])
    planet_dot.set_data([xy[-1, 0, 0]], [xy[-1, 0, 1]])
    particle_dot.set_data([xy[-1, 1, 0]], [xy[-1, 1, 1]])
    planet_trail.set_data(xy[:, 0, 0], xy[:, 0, 1])
    particle_trail.set_data(xy[:, 1, 0], xy[:, 1, 1])
    zoom_planet_trail.set_data(xy[:, 0, 0], xy[:, 0, 1])
    zoom_planet_dot.set_data([xy[-1, 0, 0]], [xy[-1, 0, 1]])
    zoom_particle_trail.set_data(xy[:, 1, 0], xy[:, 1, 1])
    zoom_particle_dot.set_data([xy[-1, 1, 0]], [xy[-1, 1, 1]])
    if now - anim["shown"] > 0.5:
        anim["shown"] = now
        set_status(f"Running: t = {anim['t']:.1f} planet orbits\nconjunction {int(orb.segment(anim['t']))}")
    # at most 4 full redraws a second, or at high speed nearly every frame would be one
    if now - anim["traced"] > 0.25 and refresh_traces():
        anim["traced"] = now
        fig.canvas.draw_idle()   # the traces are part of the background
    else:
        blit_frame()

def start_orbit(orb, color):
    anim.update(orb=orb, t=0.0, last=time.perf_counter(), paused=False, traces=None, shown=0.0, conj=[])
    set_conj_lines([])
    for art in coloured:
        art.set_color(color)
    pause_btn.label.set_text("Pause")
    refresh_traces(force=True)
    fig.canvas.draw_idle()

def stop_orbit():
    anim.update(orb=None, conj=[])
    for art in moving[:-1] + [zoom_history]:
        art.set_data([], [])
    pause_btn.label.set_text("Pause")

# ======================
# Mouse and controls
# ======================
def on_click(event):
    """Left panel: run the Namouni map and the N-body integration from the clicked point."""
    if event.inaxes != ax_sec or event.xdata is None or ax_sec.get_navigate_mode() is not None:
        return
    theta0, ar0 = -float(event.xdata), float(ar_from_ratio(event.ydata))   # conjunction angle = -theta
    if eccentricity_for(ar0) is None:
        set_status(f"Period ratio {event.ydata:.4f} is below\nthe section's C = K - I surface;\nclick higher.", 'crimson')
        blit_frame()
        return
    color = palette[len(orbits) % len(palette)]
    try:
        sec = MapOrbit(theta0, ar0)
    except RuntimeWarning as e:
        set_status(f"Namouni map failed from theta={theta0:.3f},\na_r={ar0:.3f}: {e}", 'crimson')
        blit_frame()
        return
    orbits.append(ax_sec.plot(conj_angle(sec.theta), ratio_from_ar(sec.ar), '.', color=color, markersize=3))
    print(f"Orbit {len(orbits)}: theta0={theta0:.3f}, a_r0={ar0:.3f}, period ratio={event.ydata:.4f}, "
          f"e0={eccentricity_for(ar0)/epsilon:.2f} Hill")
    set_status(f"Orbit {len(orbits)}: angle = {event.xdata:.2f},\nperiod ratio = {event.ydata:.4f}")
    start_orbit(NBodyOrbit(theta0, ar0), color)

# Middle panel: a click outside the box centres the box there; pressing inside drags it.
# Zoom panel: dragging pans the view (the box moves the opposite way), for adjustments finer
# than a pixel of the middle panel.
drag = {"mode": None}

def on_box_press(event):
    if event.button != 1 or event.xdata is None:
        return
    if event.inaxes == ax_xy and ax_xy.get_navigate_mode() is None:
        p = np.array([event.xdata, event.ydata])
        if np.any(np.abs(p - zoom["centre"]) > zoom["half"]):
            set_zoom(centre=p)
        drag.update(mode="box", offset=p - zoom["centre"])
    elif event.inaxes == ax_zoom and ax_zoom.get_navigate_mode() is None:
        drag.update(mode="pan", start=np.array([event.x, event.y]), centre=zoom["centre"].copy())

def on_box_motion(event):
    if drag["mode"] == "box" and event.inaxes == ax_xy and event.xdata is not None:
        set_zoom(centre=np.array([event.xdata, event.ydata]) - drag["offset"])
    elif drag["mode"] == "pan" and event.x is not None:
        per_pixel = 2*zoom["half"]/ax_zoom.bbox.width
        set_zoom(centre=drag["centre"] - (np.array([event.x, event.y]) - drag["start"])*per_pixel)

def on_box_release(event):
    drag["mode"] = None

def on_scroll(event):
    """Zoom panel: scroll to zoom about the cursor, so the point under it stays put."""
    if event.inaxes != ax_zoom or event.xdata is None:
        return
    h = zoom["half"]
    h_new = np.clip(h*1.5**(-event.step), zoom_min*hill, zoom_max*hill)   # scrolling up zooms in
    if h_new == h:
        return
    p = np.array([event.xdata, event.ydata])
    zoom["centre"] = p + (zoom["centre"] - p)*h_new/h
    zoom_slider.set_val(np.log10(h_new/hill))   # moves the slider; its callback applies the zoom

def on_speed(val):
    speed_slider.valtext.set_text(f"{10**val:.4g} planet orbits/s")

def on_zoom(val):
    zoom_slider.valtext.set_text(zoom_text(val))
    set_zoom(half=10**val*hill)

def toggle_pause(event=None):
    if anim["orb"] is None:
        return
    anim["paused"] = not anim["paused"]
    anim["last"] = time.perf_counter()   # no jump for the paused time
    pause_btn.label.set_text("Resume" if anim["paused"] else "Pause")
    if anim["paused"]:
        set_status(f"Paused at t = {anim['t']:.1f} planet orbits.")
    fig.canvas.draw_idle()

def clear_all(event=None):
    stop_orbit()
    for artists in orbits:
        for art in artists:
            art.remove()
    orbits.clear()
    set_status("Cleared. Click the Namouni map\nto start an orbit.")
    fig.canvas.draw_idle()

for name, fn in (('button_press_event', on_click), ('button_press_event', on_box_press),
                 ('motion_notify_event', on_box_motion), ('button_release_event', on_box_release),
                 ('scroll_event', on_scroll)):
    fig.canvas.mpl_connect(name, report_errors(fn))
fig.canvas.mpl_connect('draw_event', on_draw)
speed_slider.on_changed(report_errors(on_speed))
zoom_slider.on_changed(report_errors(on_zoom))
pause_btn.on_clicked(report_errors(toggle_pause))
clear_btn.on_clicked(report_errors(clear_all))
for w in (speed_slider, zoom_slider):
    w.label.set_fontsize(font_size - 1)
    w.valtext.set_fontsize(font_size - 1)
for b in (pause_btn, clear_btn):
    b.label.set_fontsize(font_size)

timer = fig.canvas.new_timer(interval=interval)
timer.add_callback(report_errors(tick))
fig.canvas.mpl_connect('close_event', lambda event: timer.stop())

set_zoom()
set_status("Click the Namouni map\nto start an orbit.")
timer.start()
plt.show(block=False)

Orbit 1: theta0=-0.498, a_r0=11.824, period ratio=1.1255, e0=5.59 Hill
Orbit 2: theta0=-1.967, a_r0=11.210, period ratio=1.1188, e0=4.54 Hill
Orbit 3: theta0=0.509, a_r0=11.482, period ratio=1.1218, e0=5.03 Hill
Orbit 4: theta0=0.646, a_r0=11.660, period ratio=1.1237, e0=5.33 Hill
Orbit 5: theta0=-2.906, a_r0=11.717, period ratio=1.1243, e0=5.42 Hill
Orbit 6: theta0=-3.111, a_r0=11.703, period ratio=1.1242, e0=5.40 Hill
Orbit 7: theta0=-2.633, a_r0=11.917, period ratio=1.1265, e0=5.74 Hill
Orbit 8: theta0=-2.377, a_r0=12.024, period ratio=1.1276, e0=5.90 Hill
Orbit 9: theta0=-2.155, a_r0=12.252, period ratio=1.1301, e0=6.24 Hill
Orbit 10: theta0=-1.933, a_r0=12.422, period ratio=1.1319, e0=6.49 Hill
Orbit 11: theta0=-1.489, a_r0=11.403, period ratio=1.1209, e0=4.89 Hill
Orbit 12: theta0=-1.369, a_r0=11.060, period ratio=1.1172, e0=4.26 Hill
Orbit 13: theta0=-1.421, a_r0=10.867, period ratio=1.1151, e0=3.87 Hill
Orbit 14: theta0=-1.369, a_r0=10.738, period ratio=1.1137, e0=3.59 Hill
Orb

Traceback (most recent call last):
  File "/var/folders/yv/smlvnn8n0y17w76xpsybjzfr0000gn/T/ipykernel_47522/1930097831.py", line 175, in wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/var/folders/yv/smlvnn8n0y17w76xpsybjzfr0000gn/T/ipykernel_47522/1930097831.py", line 285, in tick
    blit_frame()
  File "/var/folders/yv/smlvnn8n0y17w76xpsybjzfr0000gn/T/ipykernel_47522/1930097831.py", line 164, in blit_frame
    fig.canvas.restore_region(background["img"])
    ^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'FigureCanvasBase' object has no attribute 'restore_region'
